# <font color='blue'>SRP - Sentiment Rating Prediction</font>
## <font color='blue'>1- Fully Connected Neural Network</font>
## <font color='blue'>2- LSTM (Long Short-Term Memory)</font>
## <font color='blue'>3- BERT (Pre-Trained Transformer Model)</font>

In [ ]:
!pip install -q -U watermark

In [ ]:
# Use esta versão no ambiente local
#!pip install -q numpy==2.1.3

In [ ]:
# Use esta versão no Colab
!pip install -q numpy==2.0.2

In [ ]:
#!pip install -q spacy

In [ ]:
#!pip install -q tensorflow

In [ ]:
#!pip install -q keras

In [ ]:
#!pip install -q tf-keras

In [ ]:
#!pip install -q keras-preprocessing

In [ ]:
!pip install -q "transformers==4.44.0"

In [ ]:
import transformers
print("Versão:", transformers.__version__)

# Verificar suporte TF de outra forma
try:
    import tensorflow as tf
    print("TF versão:", tf.__version__)
except:
    print("TensorFlow: NÃO instalado")

# Ver o que está disponível no transformers
print("\nTFAuto disponível:", hasattr(transformers, 'TFAutoModel'))
print("AutoConfig disponível:", hasattr(transformers, 'AutoConfig'))

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=3

In [ ]:
# Imports — compatível com Keras 3.x / TensorFlow 2.19 / Colab
import math
import nltk
import spacy
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import transformers
from tokenizers import BertWordPieceTokenizer
from tqdm import tqdm
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from keras.utils import to_categorical, pad_sequences
from keras.models import Sequential, load_model
from keras.src.legacy.preprocessing.text import Tokenizer
from keras.metrics import Precision, Recall, AUC
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Layer
from keras.callbacks import EarlyStopping, LearningRateScheduler, CallbackList, ReduceLROnPlateau
from keras.optimizers import Adam
from keras.regularizers import l1_l2
from keras.saving import register_keras_serializable
from transformers import TFDistilBertModel, DistilBertConfig
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%reload_ext watermark
%watermark -a "Silmara Basso" -d -t -v -p numpy,spacy,tensorflow,keras,transformers

## Loading Text Data

In [ ]:
import os

# Ver diretório atual
print("Diretório atual:", os.getcwd())

# Listar tudo em /content
print("\nConteúdo de /content:")
for item in os.listdir('/content'):
    print(" ", item)

# Ver se a pasta data existe
if os.path.exists('/content/data'):
    print("\nConteúdo de /content/data:")
    for item in os.listdir('/content/data'):
        print(" ", item)
else:
    print("\nPasta 'data' não existe em /content")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
training_data_srp = pd.read_csv(
    '/content/drive/MyDrive/data/dt_training.txt',
    header=None,
    delimiter=';'
)

In [ ]:
# Load the test data
test_srp_data = pd.read_csv(
    '/content/drive/MyDrive/data/dt_test.txt',
    header=None,
    delimiter=';'
)

In [ ]:
# Adjust the column names
training_data_srp = training_data_srp.rename(columns = {0: 'message', 1: 'sentiment'})
test_srp_data = test_srp_data.rename(columns = {0: 'message', 1: 'sentiment'})

In [ ]:
# Shape
training_data_srp.shape

In [ ]:
# Shape
test_srp_data.shape

In [ ]:
# Training sample
training_data_srp.head()

The column **text** will be the input variable and **sentiment** the output variable.

In [ ]:
# Feelings present in training data
training_data_srp['sentiment'].value_counts()

In [ ]:
# Feelings present in test data
test_srp_data['sentiment'].value_counts()

## Preprocessing Text Data with SpaCy

https://spacy.io/

In [ ]:
!python -m spacy download en_core_web_md -q

In [ ]:
# Load the dictionary
srp_nlp = spacy.load('en_core_web_md')

In [ ]:
# Definition of the function 'srp_preprocess_text' that receives text as a parameter.
def srp_preprocess_text(text):

    # Process the text using the dictionary
    doc = srp_nlp(text)

    # Create a list of lemmas of the tokens, converted to lowercase and without whitespace,
    # excluding the words that are stopwords
    tokens = [token.lemma_.lower().strip() for token in doc if not token.is_stop]

    # Return the processed tokens as a single string, joining them with spaces
    return ' '.join(tokens)

In [ ]:
# Apply the function to the training data
training_data_srp['processed_text'] = training_data_srp['message'].apply(srp_preprocess_text)

In [ ]:
# Applies the function under test
test_srp_data['processed_text'] = test_srp_data['message'].apply(srp_preprocess_text)

In [ ]:
# Sample data
training_data_srp.head()

## Version 1 of the Model - Fully Connected Neural Network Architecture

### Step 1: Vectorization with TF-IDF

In [ ]:
# Create the vectorizer
srp_tfidf = TfidfVectorizer(max_df = 0.95, min_df = 2, stop_words = 'english')

In [ ]:
# Apply the vectorizer.
training_data_tfidf = srp_tfidf.fit_transform(training_data_srp['processed_text'])
test_data_tfidf = srp_tfidf.transform(test_srp_data['processed_text'])

In [ ]:
training_data_tfidf.shape

In [ ]:
type(training_data_tfidf)

In [ ]:
# Converts the input data (text) into an array.
X_training_array = training_data_tfidf.toarray()
X_test_array = test_data_tfidf.toarray()

### Step 2: Data Preparation

We now need to convert the target variable to a numeric representation. We will use Label Encoding.

In [ ]:
# Create the label encoder.
srp_read_v1 = LabelEncoder()

In [ ]:
# Fit and transform the target variable in training
y_training_read = srp_read_v1.fit_transform(training_data_srp['sentiment'])

In [ ]:
# Perform a transformation on the target variable under test
y_test_read = srp_read_v1.transform(test_srp_data['sentiment'])

We will automatically address class imbalance

In [ ]:
# Weight of classes
weights_classes = compute_class_weight('balanced', classes = np.unique(y_training_read), y = y_training_read)

In [ ]:
type(weights_classes)

In [ ]:
# Division into Training and Test (validation) Data
X_training, X_validation, y_training, y_validation = train_test_split(X_training_array,
                                                                    y_training_read,
                                                                    test_size = 0.2,
                                                                    random_state = 42,
                                                                    stratify = y_training_read)

In [ ]:
# Adjust the target variable as a categorical type
y_training_encoded = to_categorical(y_training)
y_test_encoded = to_categorical(y_test_read)
y_validation_encoded = to_categorical(y_validation)

In [ ]:
# Shape
y_training_encoded.shape, y_test_encoded.shape, y_validation_encoded.shape

### Step 3: Building the Model

In [ ]:

import sys
for name in sorted(sys.modules.keys()):
    if 'keras' in name.lower():
        print(name)

In [ ]:
from keras.regularizers import L2

# Create the model
model_srp_v1 = Sequential()

# Add the first dense layer (fully-connected) to the model
model_srp_v1.add(Dense(4096,
                        activation='selu',
                        kernel_initializer='lecun_normal',
                        input_shape=(X_training.shape[1],),
                        kernel_regularizer=L2(0.01)))

# Add the second dense layer
model_srp_v1.add(Dense(2048,
                        activation='selu',
                        kernel_initializer='lecun_normal',
                        kernel_regularizer=L2(0.01)))

# Add the third dense layer
model_srp_v1.add(Dense(1024,
                        activation='selu',
                        kernel_initializer='lecun_normal',
                        kernel_regularizer=L2(0.1)))

# Add the fourth dense layer
model_srp_v1.add(Dense(64, activation='selu'))

# Add the output layer
model_srp_v1.add(Dense(6, activation='softmax'))

### Step 4: Model Compilation and Summary

In [ ]:
# Assigns specific weights to the bias vector of the last layer of the model.
model_srp_v1.layers[-1].bias.assign(weights_classes)

In [ ]:
# Compiles the model
# Defines the optimizer as 'Adam'.
# Adam is an optimization algorithm that can be used in place of the classic stochastic gradient descent procedure
# to update the network weights iteratively based on the training data.
# Defines the loss function as 'categorical_crossentropy'. It is suitable for multiclass classification problems
# where the labels are provided in a one-hot encoded format.
# Defines the model evaluation metric as 'accuracy'.
# Accuracy is a common metric for evaluating the performance of classification models.
model_srp_v1.compile(optimizer = 'Adam',
                     loss = tf.losses.categorical_crossentropy,
                     metrics = ['accuracy', Precision(), Recall(), AUC()])

In [ ]:
model_srp_v1.summary()

### Passo 5:  Callbacks & Early Stopping

In [ ]:
# Function for scheduler parameters of the learning rate
def step_decay(epoch):
    initial_lrate = 0.001
    drop = 0.5
    epochs_drop = 10.0
    lrate = initial_lrate * math.pow(drop, math.floor((1 + epoch) / epochs_drop))
    return lrate

In [ ]:
# Learning rate scheduler
lr_scheduler = LearningRateScheduler(step_decay)

In [ ]:
# Early Stopping
early_stopping = EarlyStopping(monitor = 'val_loss', restore_best_weights = True, patience = 3)

### Step 6: Model Training

In [ ]:
# Hyperparameters
epochs_num = 20
batch_size = 256

In [ ]:
%%time
history_srp = model_srp_v1.fit(X_training,
                            y_training_encoded,
                            validation_data = (X_validation, y_validation_encoded),
                            epochs = epochs_num,
                            batch_size = batch_size,
                            callbacks = [early_stopping, lr_scheduler])

### Step 7: Model Evaluation

In [ ]:
# Extract training and validation loss
loss, val_loss = history_srp.history['loss'], history_srp.history['val_loss']

In [ ]:
# Plot
plt.plot(loss, label = 'loss')
plt.plot(val_loss, label = 'val_loss')
plt.legend()
plt.show()

In [ ]:
# Predictions based on test data
predictions_v1 = model_srp_v1.predict(X_test_array)

In [ ]:
# Extrai os labels
predictions_v1_labels = predictions_v1.argmax(axis = 1)

In [ ]:
print(classification_report(y_test_read, predictions_v1_labels))

In [ ]:
print(confusion_matrix(y_test_read, predictions_v1_labels))

In [ ]:
print(accuracy_score(y_test_read, predictions_v1_labels))

In [ ]:
# Save the template
model_srp_v1.save('model_srp_v1.keras')

### Step 8: Deploying Version 1 of the Model

In [ ]:
# Load the model
model_loaded = load_model('model_srp_v1.keras')

In [ ]:
# New sentence (feeling = fear)
phrase = "i even feel a little shaky"
df_new = pd.DataFrame({'Phrase': [phrase]})

In [ ]:
#  Applies the function under test to the new sentence
df_new['Processed_Phrase'] = df_new['Phrase'].apply(srp_preprocess_text)

In [ ]:
df_new

In [ ]:
# Apply vectorization
df_new_tfidf = srp_tfidf.transform(df_new['Processed_Phrase'])

In [ ]:
# Transform into an array
df_new_array = df_new_tfidf.toarray()

In [ ]:
# Predictions
Predictions = model_loaded.predict(df_new_array)

In [ ]:
# Select the class with the highest probability
highest_probability_class = np.argmax(Predictions, axis = 1)

In [ ]:
# Obtains the name of the class
class_name = srp_read_v1.inverse_transform(highest_probability_class)

In [ ]:
# prediction
class_name

## Version 2 of the Model - LSTM (Long Short-Term Memory)


In [ ]:
# Data processed with SpaCy
training_data_srp.head()

In [ ]:
# Creating the tokenizer
# from keras.preprocessing.text import Tokenizer
srp_tokenizer = Tokenizer()

In [ ]:
# Adjusting the tokenizer with the processed texts
srp_tokenizer.fit_on_texts(training_data_srp['processed_text'])

In [ ]:
# Extract the word indices
word_index = srp_tokenizer.word_index

In [ ]:
len(word_index)

In [ ]:
# Print the first 10 items of the word index
for i, (key, value) in enumerate(word_index.items()):
    print(key, value)
    if i == 9:
        break

In [ ]:
# Conversion of training texts to token sequences
training_sequences = srp_tokenizer.texts_to_sequences(training_data_srp['processed_text'])

In [ ]:
# Defining the maximum length of the sequences
max_length = 100

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [ ]:
# Padding of training sequences
training_sequences_padded = pad_sequences(training_sequences, maxlen = max_length, truncating = 'post')

In [ ]:
# Converting test texts into token sequences
test_sequences = srp_tokenizer.texts_to_sequences(test_srp_data['processed_text'])

In [ ]:
# Padding of test sequences
test_sequences_padded = pad_sequences(test_sequences, maxlen = max_length)

In [ ]:
# Create the label encoder
srp_read_v2 = LabelEncoder()

In [ ]:
# Fit and transform of the sentiment labels for training
y_training_read = srp_read_v2.fit_transform(training_data_srp['sentiment'])

In [ ]:
# Transforming sentiment labels into tests.
y_test_read = srp_read_v2.transform(test_srp_data['sentiment'])

In [ ]:
# Converting labels into categorical variables
y_training_encoded = to_categorical(y_training_read)
y_test_encoded = to_categorical(y_test_read)

In [ ]:
# Defining the size of the vocabulary
vocab_size = len(srp_tokenizer.word_index) + 1

In [ ]:
print(vocab_size)

In [ ]:
# Defining the dimension of the embedding
embedding_dim = max_length

In [ ]:
# Construction of the LSTM model
model_srp_v2 = tf.keras.Sequential([Embedding(vocab_size, embedding_dim, input_length = max_length),
                                     Bidirectional(LSTM(64)),
                                     Dropout(0.4),
                                     Dense(32, activation = 'leaky_relu', kernel_regularizer = l1_l2(l1 = 0.01,
                                                                                                     l2 = 0.01)),
                                     Dropout(0.4),
                                     Dense(6, activation = 'softmax')])

In [ ]:
# Model compilation
model_srp_v2.compile(loss = 'categorical_crossentropy',
                      optimizer = 'adam',
                      metrics = ['accuracy', Precision(), Recall(), AUC()])

In [ ]:
# Displaying the model summary
print(model_srp_v2.summary())

In [ ]:
# Defining the input data as an array
input_data = np.array(training_sequences_padded)

In [ ]:
# Defining the output data as an array
output_data = np.array(y_training_encoded)

In [ ]:
# Definition of hyperparameters
epochs_num = 35
validation_split_value = 0.2
patience = 5

In [ ]:
# Early Stopping Configuration
early_stopping = tf.keras.callbacks.EarlyStopping(patience = patience)

> Model Training

In [ ]:
%%time
history = model_srp_v2.fit(input_data,
                            output_data,
                            epochs = epochs_num,
                            verbose = 1,
                            validation_split = validation_split_value,
                            callbacks = [early_stopping])

In [ ]:
# Plot of error curves
loss, val_loss = history.history['loss'], history.history['val_loss']
plt.plot(loss, label = 'Error in Training')
plt.plot(val_loss, label = 'Error in Validation')
plt.legend()
plt.show()

In [ ]:
# Predictions based on test data
predictions = model_srp_v2.predict(test_sequences_padded)

In [ ]:
# Determination of predicted labels
predicted_labels = predictions.argmax(axis = 1)

In [ ]:
# Display of the classification report
print(classification_report(y_test_read, predicted_labels))

In [ ]:
# Display of the confusion matrix
print(confusion_matrix(y_test_read, predicted_labels))

In [ ]:
# Display of the model's accuracy
print(accuracy_score(y_test_read, predicted_labels))

In [ ]:
# Save the model
model_srp_v2.save('model_srp_v2.keras')

> Deploying the Model.

In [ ]:
# Load the saved model
model_loaded = load_model('model_srp_v2.keras')

In [ ]:
# New sentence (feeling = fear)
phrase = "i even feel a little shaky"

In [ ]:
# Cria um dataframe com a frase
df_new = pd.DataFrame({'Phrase': [phrase]})

In [ ]:
# Applies the processing function to the new sentence
df_new['Phrase_Processed'] = df_new['Phrase'].apply(srp_preprocess_text)

In [ ]:
# Process the new data
new_sequences = srp_tokenizer.texts_to_sequences(df_new['Phrase_Processed'])
new_sequences_padded = pad_sequences(new_sequences, maxlen = max_length)

In [ ]:
# Making predictions with the loaded model
previsoes = model_loaded.predict(new_sequences_padded)

In [ ]:
# Select the class with the highest probability
highest_probability_class = np.argmax(predictions, axis = 1)

In [ ]:
highest_probability_class

In [ ]:
# Get the class name
class_name = srp_read_v2.inverse_transform(highest_probability_class)

In [ ]:
# Predicted class
class_name

## Model Version 3 - Fine Tuning of Pre-Trained Transformer Model

I will use the data processed with SpaCy and then we will do the specific processing for the BERT model, just as we did with the model in version 1.

In [ ]:
# Data processed with SpaCy
training_data_srp.head()

In [ ]:
# Function to encode texts using the tokenizer, with specified chunk size and maximum length for truncation and padding.
# Function to encode text into integer sequences for input into the BERT model.
def srp_encode(texts, tokenizer, chunk_size = 256, maxlen = 512):

    # Configures the tokenizer to truncate text to the specified maximum length
    tokenizer.enable_truncation(max_length = maxlen)

    # Configures the tokenizer to apply padding up to the specified maximum length
    tokenizer.enable_padding(length = maxlen)

    # List to store the input IDs generated by the tokenizer.
    input_ids = []

    # List to store the attention masks generated by the tokenizer.
    attention_masks = []

    # Iterates over the texts in blocks of the size specified by chunk_size
    for i in tqdm(range(0, len(texts), chunk_size)):

        # Selects a chunk of texts to process
        text_chunk = texts[i:i+chunk_size].tolist()

        # Encodes the chunk of text in batches using the tokenizer
        encs = tokenizer.encode_batch(text_chunk)

        # Adds the encoded input IDs to the input_ids list
        input_ids.extend([enc.ids for enc in encs])

        # Adds the generated attention masks to the attention_masks list
        attention_masks.extend([enc.attention_mask for enc in encs])

    # Returns the input IDs and attention masks as numpy arrays
    return np.array(input_ids), np.array(attention_masks)

https://huggingface.co/distilbert-base-multilingual-cased

In [ ]:
# Loads the tokenizer from the pre-trained model
tokenizador_bert = transformers.DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')

In [ ]:
# Saves the tokenizer to the current directory
tokenizador_bert.save_pretrained('.')

In [ ]:
# Loads a faster tokenizer using the vocabulary of the main tokenizer
# from tokenizers import BertWordPieceTokenizer
fast_tokenizer = BertWordPieceTokenizer('vocab.txt', lowercase = False)

In [ ]:
# Visualizes the tokenizer
fast_tokenizer

In [ ]:
# Divide the data into training and validation sets using stratified sampling.
X_training, X_valid, Y_training, Y_valid = train_test_split(training_data_srp['processed_text'].values,
                                                        training_data_srp['sentiment'].values,
                                                        test_size = 0.2,
                                                        random_state = 42,
                                                        stratify = training_data_srp['sentiment'])

In [ ]:
# Maximum length used in the text encoding process, which determines the truncation and padding of the input sequences.
max_length = 100

In [ ]:
# Aplica a codificação (tokenização) em nossos dados
X_final_training, mask_training = srp_encode(X_training, fast_tokenizer, maxlen=max_length)
X_final_valid, mask_valid = srp_encode(X_valid, fast_tokenizer, maxlen=max_length)
X_final_test, mask_test = srp_encode(test_srp_data['processed_text'].to_numpy(), fast_tokenizer, maxlen=max_length)


In [ ]:
X_final_training.shape

In [ ]:
# Defines the encoder for the output data
srp_read_v3 = LabelEncoder()

In [ ]:
# Aplica o codificador (fit_transform somente nos dados de treino)
y_training_read = srp_read_v3.fit_transform(Y_training)
y_valid_read = srp_read_v3.transform(Y_valid)
y_test_read = srp_read_v3.transform(test_srp_data['sentiment'])

In [ ]:
# Converts the output variable to a categorical variable
y_training_encoded = to_categorical(y_training_read)
y_valid_encoded = to_categorical(y_valid_read)
y_test_encoded = to_categorical(y_test_read)

In [ ]:
# batch size
BATCH_SIZE = 16

In [ ]:
# Creates a TensorFlow dataset for training, which includes the input data (X_final_training and mask_training) and the corresponding labels (y_training_encoded). The dataset is configured to repeat indefinitely, shuffle the data with a buffer size of 2048, and batch the data into groups of size specified by BATCH_SIZE.
dataset_training = (
    tf.data.Dataset
    .from_tensor_slices(((X_final_training, mask_training), y_training_encoded))
    .repeat()
    .shuffle(2048)
    .batch(BATCH_SIZE)
)

In [ ]:
# Creates a TensorFlow dataset for validation, which includes the input data (X_final_valid and mask_valid) and the corresponding labels (y_valid_encoded). The dataset is configured to batch the data into groups of size specified by BATCH_SIZE and cache the data for improved performance during training.
dataset_valid = (
    tf.data.Dataset
    .from_tensor_slices(((X_final_valid, mask_valid), y_valid_encoded))
    .batch(BATCH_SIZE)
    .cache()
)

In [ ]:
# Prepare the test dataset in the format expected by TensorFlow
dataset_test = (
    tf.data.Dataset
    .from_tensor_slices(((X_final_test, mask_test), y_test_encoded))
    .batch(BATCH_SIZE)
)

In [ ]:
from keras import Input, Model

# Function to create the model using a custom Transformer layer
def srp_create_model(transformer, max_len=512):

    # Input layer for word IDs
    input_word_ids = Input(
        shape=(max_len,), dtype='int32', name="input_word_ids"
    )

    # Input layer for attention mask
    attention_mask = Input(
        shape=(max_len,), dtype='int32', name="attention_mask"
    )

    # Custom layer for the Transformer
    sequence_output = TransformerLayer(transformer)(
        [input_word_ids, attention_mask]
    )

    # Selecting the CLS token (first token)
    cls_token = sequence_output[:, 0, :]

    # Dense layer with softmax for classification
    out = Dense(6, activation="softmax")(cls_token)

    # Keras model
    model = Model(
        inputs=[input_word_ids, attention_mask], outputs=out
    )

    # Compiling the model
    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["accuracy", Precision(), Recall(), AUC()],
    )

    return model

> Loading the Pre-Trained Model:

https://huggingface.co/distilbert-base-multilingual-cased

In [ ]:
# Register the TransformerLayer class as serializable in Keras
@register_keras_serializable(package = "Custom", name = "TransformerLayer")
class TransformerLayer(Layer):

    # Initializes the TransformerLayer class with the Transformer model and other settings
    def __init__(self, transformer, **kwargs):

        # Initializes the base Layer class
        super().__init__(**kwargs)

        #  Stores the Transformer model in the class
        self.transformer = transformer

    # Defines the call operation of the layer, executed at the time of training or inference
    def call(self, inputs):

        # Unpacks the inputs (input_ids and attention_mask)
        input_ids, attention_mask = inputs

        # Executes the Transformer model
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Returns the last hidden output of the Transformer model
        return outputs.last_hidden_state

    # Defines how the layer will be serialized
    def get_config(self):

        # Obtains the configuration of the base class
        config = super().get_config()

        # Adds the complete configuration of the Transformer model
        config.update({
            "transformer_config": self.transformer.config.to_dict()  # Saves the configuration of the Transformer
        })
        return config  # Returns the complete layer configuration

    # Method to recreate the layer from a saved configuration
    @classmethod
    def from_config(cls, config):

        # Reconstructs the Transformer configuration from the saved configuration
        transformer_config = DistilBertConfig.from_dict(config["transformer_config"])

        # Loads the pre-trained Transformer model with the reconstructed configuration
        transformer = TFDistilBertModel.from_pretrained(
            "distilbert-base-multilingual-cased",
            config=transformer_config
        )

        # Returns a new instance of the TransformerLayer
        return cls(transformer=transformer)

In [ ]:
# Loading the pre-trained model
transformer_layer = TFDistilBertModel.from_pretrained(
    "distilbert-base-multilingual-cased"
)

In [ ]:
#   Creating the model using the custom Transformer layer and specifying the maximum sequence length for truncation and padding of the input data.
model_srp_v3 = srp_create_model(transformer_layer, max_len = max_length)


In [ ]:
# Displaying the model summary
model_srp_v3.summary()

In [ ]:
# Setting the first three layers of the model to non-trainable
model_srp_v3.layers[0].trainable = False
model_srp_v3.layers[1].trainable = False
model_srp_v3.layers[2].trainable = False

In [ ]:
#   summary
model_srp_v3.summary()

In [ ]:
#  Defining the number of steps per epoch based on the size of the training data and the batch size, and setting the number of epochs for training.
n_steps = X_final_training.shape[0] // BATCH_SIZE
epochs_num = 3

>  Model Training

In [ ]:
%%time
history = model_srp_v3.fit(dataset_training,
                            steps_per_epoch = n_steps,
                            validation_data = dataset_valid,
                            epochs = epochs_num)

In [ ]:
# Plot of error curves
loss, val_loss = history.history['loss'], history.history['val_loss']
plt.plot(loss, label = 'Training error')
plt.plot(val_loss, label = 'Validation error')
plt.legend()
plt.show()

In [ ]:
# Applying the encoding function to the test data to prepare it for prediction with the trained model.
X_final_test, mask_test = srp_encode(
    test_srp_data['processed_text'].to_numpy(),
    fast_tokenizer,
    maxlen = max_length
)

# Perform the forecast using both tensors
predictions = model_srp_v3.predict((X_final_test, mask_test))

In [ ]:
# Labels
labels_predictions = predictions.argmax(axis = 1)

In [ ]:
print(classification_report(y_test_read, labels_predictions))

In [ ]:
print(confusion_matrix(y_test_read, labels_predictions))

In [ ]:
print(accuracy_score(y_test_read, labels_predictions))

In [ ]:
# Save the model
model_srp_v3.save("model_srp_v3.keras")

> Deploying the Model

In [ ]:
# Load the model

# Imports
from transformers import TFDistilBertModel
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import custom_object_scope

model_reloaded = load_model(
    "model_srp_v3.keras",
    custom_objects = {"TransformerLayer": TransformerLayer}
)

In [ ]:
# New sentence (feeling = fear)
phrase = "i even feel a little shaky"

In [ ]:
# Creates a dataframe with the sentence
df_new = pd.DataFrame({'Phrase': [phrase]})

In [ ]:
# Applies the processing function to the new sentence
df_new['Phrase_Processed'] = df_new['Phrase'].apply(srp_preprocess_text)

In [ ]:
new_data = srp_encode(df_new['Phrase_Processed'], fast_tokenizer, maxlen = max_length)

In [ ]:
# Making predictions with the loaded model
predictions = model_srp_v3.predict(new_data)

In [ ]:
# Select the class with the highest probability
highest_probability_class = np.argmax(predictions, axis = 1)

In [ ]:
highest_probability_class

In [ ]:
# Get the class name
class_name = srp_read_v3.inverse_transform(highest_probability_class)

In [ ]:
# Get the class name
class_name

In [ ]:
# To avoid parallelism issues with tokenizers, we set the environment variable TOKENIZERS_PARALLELISM to false.
%env TOKENIZERS_PARALLELISM=false

In [ ]:
%watermark -a "Silmara Basso"

In [ ]:
%watermark -v -m

In [ ]:
%watermark --iversions